# Alaska Range: Our product vs Passive Microwave (Pan et al., 2021)

Here we compare against against an independent passive microwave melt onset product ([ORNL DAAC: Main Melt Onset Dates, 1841](https://www.earthdata.nasa.gov/data/catalog/ornl-cloud-main-melt-onset-dates-1841-1.0#documents-and-resources)) (6.25km resolution) over the Alaska Range for the 2020 water year.

In [ ]:
import xarray as xr
import rioxarray as rxr
from global_snowmelt_runoff_onset.config import Config, Tile
from global_snowmelt_runoff_onset.plot_utils import create_month_colorbar, create_diverging_colorbar
import easysnowdata
import glob
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import contextily as ctx
from matplotlib_scalebar.scalebar import ScaleBar

import matplotlib.gridspec as gridspec
import matplotlib.patheffects as path_effects
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import ConnectionPatch
from matplotlib_scalebar.scalebar import ScaleBar
import shapely

In [ ]:
from pathlib import Path

config = Config('config/global_config_v10.txt')

# Figures are scoped by dataset version so a v10 run cannot overwrite the v9 figures.
# Switching versions = editing the config path above.
VERSION = config.version
FIGURE_DIR = Path('figures') / VERSION
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
passive_files = glob.glob(f"data/Main_Melt_Onset_Dates_1841/data/*1.tif")
years = [int(f.split('.')[2][:4]) for f in passive_files]
ds_list = []
da_template = rxr.open_rasterio(passive_files[0], masked=True).squeeze()
for f, year in zip(passive_files, years):
    da = rxr.open_rasterio(f, masked=True).squeeze()
    da = da.rename('melt_onset_dayofyear')
    da = da.rio.reproject_match(da_template)
    da = da.assign_coords(year=year)
    ds_list.append(da.to_dataset())
    
passive_ds = xr.concat(ds_list, dim='year').sortby('year')
passive_ds['melt_onset_mean'] = passive_ds['melt_onset_dayofyear'].mean(dim='year')
passive_ds['melt_onset_median'] = passive_ds['melt_onset_dayofyear'].median(dim='year')
passive_ds['melt_onset_std'] = passive_ds['melt_onset_dayofyear'].std(dim='year')
passive_ds['melt_onset_anomaly'] = passive_ds['melt_onset_dayofyear'] - passive_ds['melt_onset_median']
mask_file = glob.glob(f"data/Main_Melt_Onset_Dates_1841/data/*mask*.tif")
mask_da = rxr.open_rasterio(mask_file[0], masked=True).squeeze()
passive_ds["mask"] = mask_da
passive_ds

In [ ]:
global_ds = config.open_runoff_onset_dataset()
global_ds

In [ ]:
url = (f"https://data.earthenv.org/mountains/standard/GMBA_Inventory_v2.0_standard_300.zip")
gmba_gdf = gpd.read_file("zip+" + url)
alaska_range_gdf = gmba_gdf[gmba_gdf["MapName"]=="Alaska Range"]
xmin_4326, ymin_4326, xmax_4326, ymax_4326 = alaska_range_gdf.total_bounds
alaska_range_gdf

In [ ]:
alaska_proj = "EPSG:3338"
alaska_range_proj_gdf = alaska_range_gdf.to_crs(alaska_proj)
xmin_proj, ymin_proj, xmax_proj, ymax_proj = alaska_range_proj_gdf.total_bounds
alaska_range_proj_gdf

In [ ]:
xlim_min = xmin_proj - 1.00e5
xlim_max = xmax_proj + 1.00e5
ylim_min = ymin_proj - 1.00e5
ylim_max = ymax_proj + 1.30e5
clipbox = [xlim_min, ylim_min, xlim_max, ylim_max]

In [ ]:
states_gdf = gpd.read_file('http://eric.clst.org/assets/wiki/uploads/Stuff/gz_2010_us_040_00_5m.json')
alaska_gdf = states_gdf[states_gdf['NAME']=='Alaska'].clip_by_rect(-175, 0, 0, 89)
alaska_proj_gdf = alaska_gdf.to_crs(alaska_proj)
hillshade_proj_da = rxr.open_rasterio('../../visualize/data/global_hillshade_robinson.tif', masked=True, chunks='auto').squeeze().rio.clip_box(*alaska_gdf.total_bounds,crs=alaska_gdf.crs).rio.reproject(alaska_proj)#.coarsen(x=4, y=4,boundary='trim').mean()#.compute()
hillshade_proj_da

In [ ]:
alaska_passive_onset_2020_proj_da = passive_ds['melt_onset_dayofyear'].sel(year=2020).rio.reproject(alaska_proj_gdf.crs)#.rio.clip(alaska_proj_gdf.geometry)#.plot.imshow()
alaska_passive_onset_2020_proj_da

In [ ]:
alaska_runoff_onset_2020_proj_da = global_ds['runoff_onset'].sel(water_year=2020).rio.clip_box(xmin_4326-2.2, ymin_4326-2.2, xmax_4326+4, ymax_4326+4, crs="EPSG:4326").coarsen(latitude=4,longitude=4, boundary='trim').mean().rio.reproject(alaska_proj).rio.clip_box(*clipbox,crs=alaska_proj)
alaska_runoff_onset_2020_proj_da

In [ ]:
differences_proj_da = alaska_runoff_onset_2020_proj_da - (passive_ds['melt_onset_dayofyear'].sel(year=2020).rio.reproject_match(alaska_runoff_onset_2020_proj_da)+92).rio.clip_box(*clipbox,crs=alaska_proj)
differences_proj_da

In [ ]:
# --- Color / date range setup ---
date_vmin = "2020-01-01"
date_vmax = "2020-07-31"
doy_vmin  = pd.to_datetime(date_vmin).dayofyear
doy_vmax  = pd.to_datetime(date_vmax).dayofyear
dowy_vmin = easysnowdata.utils.datetime_to_DOWY(date_vmin)
dowy_vmax = easysnowdata.utils.datetime_to_DOWY(date_vmax)

print(f'for vmin date {date_vmin}, doy={doy_vmin}, dowy={dowy_vmin}')
print(f'for vmax date {date_vmax}, doy={doy_vmax}, dowy={dowy_vmax}')

# Shared style constants
TITLE_FS = 28
STROKE   = [path_effects.withStroke(linewidth=3, foreground='black')]

# ── Helpers ───────────────────────────────────────────────────────────────
# Draw background layers only — call _finalize_map_ax after all data layers.
def _style_map_ax(ax):
    hillshade_proj_da.coarsen(x=4, y=4, boundary='trim').mean().plot.imshow(
        ax=ax, cmap='gray', vmin=0, vmax=255, add_colorbar=False, zorder=0)
    alaska_range_proj_gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=1.5, zorder=2)

# Apply limits, aspect, and axis visibility after all imshow calls are done.
# imshow resets xlim/ylim/aspect, so this must run last.
def _finalize_map_ax(ax):
    ax.set_xlim([xlim_min, xlim_max])
    ax.set_ylim([ylim_min, ylim_max])
    ax.set_aspect('equal')
    ax.axis('off')

def _map_title(ax, text):
    ax.text(0.5, 0.98, text, transform=ax.transAxes,
            ha='center', va='top', color='white', fontname='Oswald',
            fontsize=TITLE_FS, fontweight='bold', path_effects=STROKE)

# --- Figure / GridSpec ---
fig = plt.figure(figsize=(20, 10))
outer_gs = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[1.5, 1.6, 1.6],
                              wspace=0.04, left=0.01, right=0.99, bottom=0.03, top=0.97)

# Globe fills the entire left column
ax_globe = fig.add_subplot(outer_gs[0, 0],
                            projection=ccrs.Orthographic(central_longitude=-150, central_latitude=64))


pos = ax_globe.get_position()
scale = 0.78
new_w = pos.width * scale
new_h = pos.height * scale
ax_globe.set_position([
    pos.x0 + (pos.width - new_w) / 2,   # keep centered horizontally
    pos.y0 + (pos.height - new_h) * 0.9, # push toward top
    new_w, new_h
])


left_bbox = outer_gs[0, 0].get_position(fig)
sat_frac  = 0.55
ax_sat = fig.add_axes([left_bbox.x0,
                        left_bbox.y0,
                        left_bbox.width,
                        left_bbox.height * sat_frac])

mid_gs     = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=outer_gs[0, 1],
                                               height_ratios=[1, 0.1, 1], hspace=0.00)
ax_passive = fig.add_subplot(mid_gs[0])
ax_cbar    = fig.add_subplot(mid_gs[1])
ax_ours    = fig.add_subplot(mid_gs[2])

right_gs     = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=outer_gs[0, 2],
                                                 height_ratios=[1, 0.1, 1], hspace=0.00)
ax_diff      = fig.add_subplot(right_gs[0])
ax_diff_cbar = fig.add_subplot(right_gs[1])
ax_hist      = fig.add_subplot(right_gs[2])

_p = ax_hist.get_position()
#ax_hist.set_position([_p.x0 + 0.025, _p.y0, _p.width - 0.025, _p.height - 0.035])
ax_hist.set_position([_p.x0 + 0.025, _p.y0+0.05, _p.width - 0.03, _p.height - 0.13])


# ── Colorbars first — create_month_colorbar calls fig.canvas.draw() internally
# ── which triggers a layout pass; draw before any map axis is configured.

# ── 4. Shared viridis colorbar (month-slot labels, dashed boundaries) ──────
create_month_colorbar(dowy_vmin, dowy_vmax, ax=ax_cbar, hemisphere='northern',
                      cmap='viridis', label='', # Melt timing [Day of water year]
                      tick_labelsize=0, label_fontsize=0, month_fontsize=16)

# ── 7. Diverging colorbar ─────────────────────────────────────────────────
create_diverging_colorbar(-75, 75, ax=ax_diff_cbar, cmap='RdBu',
                          label='Difference [days]', ticks=[-75, -50, -25, 0, 25, 50, 75],
                          minor_tick_spacing=25, left_text='Our product earlier',
                          right_text='Our product later', label_fontsize=14, tick_labelsize=14,
                          text_fontsize=16)

# ── 1. Orthographic globe ──────────────────────────────────────────────────
ax_globe.set_global()
ax_globe.add_feature(cfeature.LAND, zorder=0, edgecolor='gray', linewidth=0.3)
ax_globe.add_feature(cfeature.OCEAN, zorder=0)
ax_globe.add_geometries(alaska_range_gdf.geometry, crs=ccrs.PlateCarree(),
                         edgecolor='black', facecolor='none', linewidth=0.3, zorder=2)
bbox_4326 = shapely.geometry.box(xmin_4326 - 1, ymin_4326 - 1, xmax_4326 + 1, ymax_4326 + 1)
ax_globe.add_geometries([bbox_4326], crs=ccrs.PlateCarree(),
                         edgecolor='red', facecolor='none', linewidth=1, zorder=3)

# ── 2. Satellite / basemap context ────────────────────────────────────────
alaska_range_proj_gdf.boundary.plot(ax=ax_sat, edgecolor='black', linewidth=1.5, zorder=2)
ctx.add_basemap(ax_sat, crs=alaska_range_proj_gdf.crs.to_epsg(),
                source=ctx.providers.Esri.WorldImagery, attribution='', zoom=6)
ax_sat.set_xlim([xlim_min, xlim_max])
ax_sat.set_ylim([ylim_min, ylim_max])
ax_sat.set_aspect('equal')
ax_sat.axis('off')
scalebar = ScaleBar(1, 'm', fixed_value=150, fixed_units='km', location='upper right',
                    color='white', box_color='none', box_alpha=0,
                    font_properties={'size': 14, 'weight': 'bold'})
ax_sat.add_artist(scalebar)
_map_title(ax_sat, 'Alaska Range')
ax_sat.set_title("")

# ── Red connecting lines: globe bbox bottom corners → satellite top corners ─
pt_ll = ax_globe.projection.transform_point(xmin_4326 - 1, ymin_4326 - 1, ccrs.PlateCarree())
pt_lr = ax_globe.projection.transform_point(xmax_4326 + 1, ymin_4326 - 1, ccrs.PlateCarree())
for xyA, xyB in [(pt_ll, (xlim_min, ylim_max)),
                 (pt_lr, (xlim_max, ylim_max))]:
    fig.add_artist(ConnectionPatch(xyA=xyA, coordsA=ax_globe.transData,
                                   xyB=xyB, coordsB=ax_sat.transData,
                                   color='red', linewidth=1, zorder=10))

# ── 3. Passive microwave melt onset ───────────────────────────────────────
_style_map_ax(ax_passive)
alaska_passive_onset_2020_proj_da.plot.imshow(ax=ax_passive, cmap='viridis',
                                       vmin=doy_vmin, vmax=doy_vmax,
                                       add_colorbar=False, zorder=1, alpha=0.85)
_finalize_map_ax(ax_passive)
_map_title(ax_passive, 'Passive microwave melt onset, 2020')
ax_passive.set_title("")

# ── 5. Our runoff onset product ───────────────────────────────────────────
_style_map_ax(ax_ours)
alaska_runoff_onset_2020_proj_da.coarsen(x=2, y=2, boundary='trim').mean().plot.imshow(
    ax=ax_ours, cmap='viridis', vmin=dowy_vmin, vmax=dowy_vmax,
    add_colorbar=False, zorder=1, alpha=0.85)
_finalize_map_ax(ax_ours)
_map_title(ax_ours, 'Our runoff onset product, 2020')
ax_ours.set_title("")

# ── 6. Difference map ─────────────────────────────────────────────────────
norm_diff = mcolors.Normalize(vmin=-75, vmax=75)
_style_map_ax(ax_diff)
differences_proj_da.coarsen(x=2, y=2, boundary='trim').mean().plot.imshow(
    ax=ax_diff, cmap='RdBu', vmin=-75, vmax=75,
    add_colorbar=False, zorder=1, alpha=0.85)
_finalize_map_ax(ax_diff)
_map_title(ax_diff, 'Difference map (our product - passive)')
ax_diff.set_title("")

# ── 8. Histogram ──────────────────────────────────────────────────────────
diff_vals = differences_proj_da.values.flatten()
diff_vals = diff_vals[~np.isnan(diff_vals)]
n_h, bins_h, patches_h = ax_hist.hist(diff_vals, bins=50, edgecolor='black', linewidth=0.5)
cmap_rdbu = cm.get_cmap('RdBu')
for patch, l, r in zip(patches_h, bins_h[:-1], bins_h[1:]):
    patch.set_facecolor(cmap_rdbu(norm_diff((l + r) / 2)))
ax_hist.axvline(0, color='black', linestyle='--', linewidth=1.5)
ax_hist.set_xlim([-150, 150])
ax_hist.tick_params(axis='both', which='major', labelsize=14, length=6, width=1.2)
ylim_h = ax_hist.get_ylim()
ax_hist.text(-145, ylim_h[1] * 0.3, 'Our product\nearlier',
             color='darkred',  fontsize=16, fontweight='bold', ha='left')
ax_hist.text( 145, ylim_h[1] * 0.3, 'Our product\nlater',
             color='darkblue', fontsize=16, fontweight='bold', ha='right')
median_diff = np.nanmedian(diff_vals)
mad_diff    = np.nanmedian(np.abs(diff_vals - median_diff))
ax_hist.text(145, ylim_h[1] * 0.9,
             f'Median difference: {median_diff:.1f} days\nMAD: {mad_diff:.1f} days',
             fontsize=13, ha='right', va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax_hist.set_xlabel('Difference [days]', fontsize=14)
ax_hist.set_ylabel('Count', fontsize=14)
ax_hist.spines['top'].set_visible(False)
ax_hist.spines['right'].set_visible(False)
 
fig.savefig(FIGURE_DIR / 'alaska_range_passive_comparison.png', dpi=300, bbox_inches='tight')

In [ ]:
# Persist the Fig. 6 stats -- until now they existed only as the text annotation
# drawn on the figure itself.
from global_snowmelt_runoff_onset.results import save_result_table

passive_stats_df = pd.DataFrame([{
    'region': 'Alaska Range',
    'water_year': 2020,
    'comparison': 'ours_minus_passive_microwave_Pan_et_al_2021',
    'n_pixels': int(diff_vals.size),
    'median_difference_days': round(float(median_diff), 1),
    'mad_days': round(float(mad_diff), 1),
}])
save_result_table(passive_stats_df, 'passive_comparison_stats', version=VERSION)
passive_stats_df